In [27]:
%pip install matplotlib


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
import matplotlib.pyplot as plt
import math
import sys

# Complete inference

## Tokenizer

In [29]:
import json
import regex

class Tokenizer:

    def __init__(self, tokenizer_path: str):
        with open(tokenizer_path) as f:
            tokenizer_data = json.load(f)
        split = next(filter(lambda t: t["type"] == "Split", tokenizer_data["pre_tokenizer"]["pretokenizers"]))
        self.split_regex = regex.compile(split["pattern"]["Regex"])

        # space is encoded as Ġ, for simplicity, just use space here.
        self.vocab = {k.replace("Ġ", " ").encode("utf-8"): v for k, v in tokenizer_data["model"]["vocab"].items()}
        added_tokens = {t["content"]: t["id"] for t in tokenizer_data["added_tokens"]}

        self.begin_of_text = added_tokens["<|begin_of_text|>"]
        self.end_of_text = added_tokens["<|end_of_text|>"]

        self.vocab.update(added_tokens)

        # inverse vocabulary for detokenization
        self.vocab_inv = { v: k for k, v in self.vocab.items() }

    def tokenize(self, text: str) -> list[int]:
        str_tokens = self.split_regex.findall(text)

        # Add specific markers for beginning and end of text
        # str_tokens = ["<|begin_of_text|>"] + str_tokens + ["<|end_of_text|>"]

        tokens = []

        for str_token in str_tokens:
            parts = [bytes([b]) for b in str_token.encode("utf-8")]

            while True:
                # Iterate over all pairs and find the pair we want to merge the most
                min_idx = None
                min_rank = None
                for i, pair in enumerate(zip(parts[:-1], parts[1:])):
                    rank = self.vocab.get(pair[0] + pair[1])
                    if rank is not None and (min_rank is None or rank < min_rank):
                        min_idx = i
                        min_rank = rank

                # If there were no pairs we could merge, we're done!
                if min_rank is None:
                    break
                assert min_idx is not None

                # Otherwise, merge that pair and leave the rest unchanged. Then repeat.
                parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2 :]

            tokens.extend(self.vocab[part] for part in parts)

        return [self.begin_of_text] + tokens
    
    def detokenize(self, tokens: list[int]) -> str:
        decoded = b""
        for t in tokens:
            decoded += self.vocab_inv[t]

        return str(decoded, "utf-8")

## Model weights

Ensure that the content of `1-fetch-files.ipynb` has been executed to fetch and convert the model weights from Huggingface

In [30]:
from pathlib import Path
import mmap
import numpy as np
from numpy.typing import NDArray

def load_raw_model(path: str):
    model_dir = Path(path)
    model = {}
    with open(model_dir / "metadata.json") as file:
        metadata = json.load(file)
    for tensor_name, tensor_metadata in metadata.items():
        if tensor_name == "__metadata__":
            continue
        file = open(model_dir / f"{tensor_name}.raw", mode="rb")
        mmaped = mmap.mmap(file.fileno(), 0, prot=mmap.PROT_READ)
        model[tensor_name] = np.frombuffer(mmaped, dtype=np.float32).reshape(tensor_metadata["shape"])
    return model

## Network blocks

### Helper functions

In [31]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [32]:
def silu(x):
    """
    Sigmoid Linear Unit
    The SiLU function is also known as the swish function.
    """
    return x * (1 / (1 + np.exp(-x)))

### Attention block

In [ ]:
from typing import Any


class AttentionBlock:
    
    def __init__(self, layer_index: int, config: dict[str, Any], weights: dict[str, NDArray], freqs_cos: NDArray, freqs_sin: NDArray):
        self.freqs_cos = freqs_cos
        self.freqs_sin = freqs_sin

        self.num_key_value_heads = config["num_key_value_heads"]
        self.num_attention_heads = config["num_attention_heads"]
        self.head_dim = config["head_dim"]
        self.max_seq_len = config["max_seq_len"]

        self.q_weight = weights[f"model.layers.{layer_index}.self_attn.q_proj.weight"].T
        self.k_weight = weights[f"model.layers.{layer_index}.self_attn.k_proj.weight"].T
        self.v_weight = weights[f"model.layers.{layer_index}.self_attn.v_proj.weight"].T
        self.o_weight = weights[f"model.layers.{layer_index}.self_attn.o_proj.weight"].T
        
        self.k_cache = np.zeros((self.max_seq_len, self.num_key_value_heads, self.head_dim))
        self.v_cache = np.zeros((self.max_seq_len, self.num_key_value_heads, self.head_dim))


    def apply_rotary_positional_encoding(self, xq: NDArray, xk: NDArray, start_pos: int) -> NDArray:
        input_length = xq.shape[0]

        freqs_cos = self.freqs_cos[start_pos: start_pos + input_length]
        freqs_sin = self.freqs_sin[start_pos: start_pos + input_length]

        # split the last dimension of the embedding in two channels to produce real, imaginary pairs.

        ### TODO: this isn't the same operation as rotate_half of the tf version
        ### r and i parts are contiguous, in tf, they are separated by x.shape[-1] // 2
        
        # Split xq and xk as a complex number representation
        xq_r, xq_i = np.split(xq, 2, axis=-1)
        xk_r, xk_i = np.split(xk, 2, axis=-1)

        freqs_cos = np.expand_dims(freqs_cos, axis=1)
        freqs_sin = np.expand_dims(freqs_sin, axis=1)

        # Apply rotation using real numbers.
        xq_out_r = xq_r * freqs_cos - xq_i * freqs_sin
        xq_out_i = xq_r * freqs_sin + xq_i * freqs_cos
        xk_out_r = xk_r * freqs_cos - xk_i * freqs_sin
        xk_out_i = xk_r * freqs_sin + xk_i * freqs_cos

        xq_out = np.concatenate([xq_out_r, xq_out_i], axis=-1)
        xk_out = np.concatenate([xk_out_r, xk_out_i], axis=-1)

        return xq_out, xk_out


    def __call__(self, x: NDArray, start_pos: int, mask: NDArray | None) -> NDArray:
        # This code isn't batched, so `x` contains only a single token

        input_length = x.shape[0]

        xq = x @ self.q_weight
        xk = x @ self.k_weight
        xv = x @ self.v_weight

        xq = xq.reshape((input_length, self.num_attention_heads, self.head_dim))
        xk = xk.reshape((input_length, self.num_key_value_heads, self.head_dim))
        xv = xv.reshape((input_length, self.num_key_value_heads, self.head_dim))

        xq, xk = self.apply_rotary_positional_encoding(xq, xk, start_pos)
        
        # Populate KV cache
        self.k_cache[start_pos: start_pos + input_length] = xk
        self.v_cache[start_pos: start_pos + input_length] = xv

        # Extract all key and values up to the current one from the cache.
        ks = self.k_cache[: start_pos + input_length]
        vs = self.v_cache[: start_pos + input_length]

        repeats = self.num_attention_heads // self.num_key_value_heads
        xk = np.repeat(ks, repeats, axis=1)
        xv = np.repeat(vs, repeats, axis=1)

        # ["L, HN, HD"] -> ["HN, L, HD"]
        xq = xq.transpose(1, 0, 2)
        xk = xk.transpose(1, 0, 2)
        xv = xv.transpose(1, 0, 2)

        # flip the last dimensions to allow matrix multiplication
        xk = xk.transpose(0, 2, 1)
        attention = xq @ xk
        attention = attention / math.sqrt(self.head_dim)

        # Mask is only used at the beginning when processing the input tokens
        if mask is not None:
            attention = attention + mask[None, :, :]
        attention = softmax(attention)

        output = attention @ xv

        # ["HN, L or 1, HD"] -> ["L or 1, D"]
        output = output.transpose(1, 0, 2).reshape(input_length, -1)
        output = output @ self.o_weight

        return output


### Other blocks

In [34]:
class FeedForward:
    def __init__(self, layer_index: int, weights: dict[str, NDArray]):
        self.up_weight = weights[f"model.layers.{layer_index}.mlp.up_proj.weight"].T
        self.down_weight = weights[f"model.layers.{layer_index}.mlp.down_proj.weight"].T
        self.gate_weight = weights[f"model.layers.{layer_index}.mlp.gate_proj.weight"].T

    def __call__(self, x: NDArray):
        swish = silu(x @ self.gate_weight)
        x_V = x @ self.up_weight
        x = swish * x_V
        x = x @ self.down_weight
        return x

In [35]:
class RMSNorm:

    def __init__(self, weights_array: NDArray, eps: float):
        self.weights = weights_array
        self.eps = eps

    def __call__(self, x: NDArray):
        x_squared = x ** 2
        rms = np.sqrt(x_squared.mean(-1, keepdims=True) + self.eps)
        rms_norm = (x / rms) * self.weights
        return rms_norm

### Transformer block

In [36]:
class TransformerBlock:
    def __init__(self, layer_index: int, config: dict[str, Any], weights: dict[str, NDArray], freqs_cos: NDArray, freqs_sin: NDArray):
        self.attention = AttentionBlock(layer_index, config, weights, freqs_cos, freqs_sin)

        self.feed_forward = FeedForward(layer_index, weights)

        self.input_layernorm = RMSNorm(
            weights.get(f"model.layers.{layer_index}.input_layernorm.weight"),
            eps=config["rms_norm_eps"]
        )
        self.post_attention_layernorm = RMSNorm(
            weights.get(f"model.layers.{layer_index}.post_attention_layernorm.weight"),
            eps=config["rms_norm_eps"]
        )

    def __call__(self, x: NDArray, start_pos: int, mask: NDArray):
        # RMSNorm
        norm_x = self.input_layernorm(x)

        # Masked Multi-Head Attention
        h1 = self.attention(norm_x, start_pos, mask)

        z = x + h1

        # RMSNorm
        norm_z = self.post_attention_layernorm(z)
        # Feed Forward + SwiGLU
        h2 = self.feed_forward(norm_z)
        out = z + h2

        return out

# Llama network

In [37]:

class Llama:
    def __init__(self, model_path: str, config_path: str, max_seq_len: int):
        weights = load_raw_model(model_path)

        with open(config_path) as file:
            config = json.load(file)

        config["max_seq_len"] = max_seq_len

        self.token_embeddings = weights.get("model.embed_tokens.weight")

        # RoPE #1
        freqs_cos, freqs_sin = self.compute_cos_sin_cache(
            config["hidden_size"] // config["num_attention_heads"],
            config["max_seq_len"],
            config["rope_theta"],
        )

        self.layers = []
        for layer_index in range(config["num_hidden_layers"]):
            self.layers.append(TransformerBlock(layer_index, config, weights, freqs_cos, freqs_sin))

        self.norm = RMSNorm(weights.get("model.norm.weight"), eps=config["rms_norm_eps"])
        self.lm_head_weight = weights.get("model.embed_tokens.weight").T


    def compute_cos_sin_cache(self, head_dim: int, max_seq_len: int, base):
        inv_freq = 1.0 / (base ** (np.arange(0, head_dim, 2)[: (head_dim // 2)] / head_dim))
        t = np.arange(max_seq_len)
        freqs = np.outer(t, inv_freq)

        return np.cos(freqs), np.sin(freqs)


    def __call__(self, input_ids, start_pos: int):
        input_length = input_ids.shape[0]
        h = self.token_embeddings[input_ids]

        # `mask` is generated only once at the beginning.
        mask = None
        if input_length > 1:
            mask = np.full((input_length, input_length), float("-inf"))
            mask = np.triu(mask, k=1)
            mask = np.concatenate([np.zeros((input_length, start_pos)), mask], axis=1)

        # Transformer Layers
        for i, layer in enumerate(self.layers):
            h = layer(h, start_pos, mask)


        # RMSNorm
        h = self.norm(h)
        # Only forward the output from the last position.
        # ["B, 1, VS"] = ["B, 1(L), D"] @ ["D, VS"]
        logit = h[[-1], :] @ self.lm_head_weight
        return logit

    def generate(self, input_ids, max_new_tokens: int):
        input_length = input_ids.shape[0]
        for i, curr_pos in enumerate(range(0, input_length + max_new_tokens)):
            if i == 0:  # Prefill Phase
                inputs = input_ids[curr_pos]
                pos = 0
            else:  # Decode Phase
                inputs = next_id
                pos = curr_pos
            logits = self(inputs, pos)
            next_id = logits[-1, :].argmax(-1, keepdims=True)
            yield next_id



In [38]:
tokenizer = Tokenizer("models/Llama-3.2-1B/tokenizer.json")
llama = Llama("models/Llama-3.2-1B/tensors-fp32", "models/Llama-3.2-1B/config.json", 200)

In [39]:
prompt = "Holy cow, it's working"

In [40]:
input_ids = np.array([tokenizer.tokenize(prompt)])[0]

In [41]:
display(input_ids.shape)
display(input_ids)

(7,)

array([128000,  72291,  19923,     11,    433,    596,   3318])

In [42]:
print(prompt, end="")

for id in llama.generate(input_ids, max_new_tokens=150):
    # L += 1µµ
    output_id = id[0].tolist()
    # if output_id[-1] in [tokenizer.eos_id, tokenizer.bos_id]:
    #     break
    # print(output_id, end="")
    print(tokenizer.detokenize([output_id]), end="")
    sys.stdout.flush()


Holy cow, it's working

IndexError: tuple index out of range